# Task 4.10 — Geo Mean HAI Across Vaccine Strains (D365, Durability)

**4.10 Predict antibody durability - all 3 vaccine strains (D365)**
* Training Data: Demographics + Day 0 + Day 7 innate + **Day 28**
* Assay: HAI / Measure: Geo mean HAI / Metric: Spearman correlation
* Full description: Geo mean of HAI across the 3 vaccine strains at Day 365

> **Note — missing strains:** only `Vic B/Austria/1359417/2021` has D365 measurements in the training data. H1N1 A/Victoria/4897/2022 stops at D28; H3N2 A/Massachusetts/18/2022 is absent entirely. This task collapses to predicting Vic B D365 — effectively the same target as Task 4.9.

---

**Target:** arithmetic mean of log2 HAI at D365 across available vaccine strains = log2(geometric mean) on the raw titer scale. `np.exp2` is applied to predictions before saving.

**Features:** all columns available at D0 + D7 + D28 (durability task — D28 is allowed). Only D365 columns excluded.

**CV:** 5-fold; Spearman correlation on holdout predictions.

In [ ]:
VACCINE_STRAINS = [
    'H1N1 A/Victoria/4897/2022',
    'H3N2 A/Massachusetts/18/2022',
    'Vic B/Austria/1359417/2021',
]
AUTO_ML_MAX_RUNTIME_SECONDS = 1200

In [ ]:
PARQUET_PATH = '../merged_data/combined.parquet'
CHALLENGE_DATA_PATH = '../cleaned_data'
SUBMISSION_PATH = '../automl_submission'

In [ ]:
import io
import os
import tempfile
import warnings
from contextlib import redirect_stderr, redirect_stdout

import h2o
import numpy as np
import pandas as pd
from h2o.automl import H2OAutoML
from scipy.stats import spearmanr

warnings.filterwarnings('ignore', category=UserWarning, module='h2o')
h2o.init()

In [ ]:
data = h2o.import_file(PARQUET_PATH)
print(f'Training data shape: {data.shape}')

challenge_participants = pd.read_csv(CHALLENGE_DATA_PATH + '/challenge_participants_cleaned.csv')
challenge_hai = pd.read_csv(CHALLENGE_DATA_PATH + '/challenge_hai_cleaned.csv')
challenge_data = challenge_hai.merge(challenge_participants, on='participant_id', how='inner')
print(f'Challenge shape: {challenge_data.shape}')

In [ ]:
# Compute target in pandas and round-trip via parquet so H2O reads NaN as H2O NA
hai_d365_cols = [c for c in data.columns if c.startswith('HAI_') and c.endswith('_d365')]
vaccine_d365_cols = [c for c in hai_d365_cols if any(s in c for s in VACCINE_STRAINS)]

pd_hai_d365 = data[hai_d365_cols].as_data_frame()
target_series = pd_hai_d365[vaccine_d365_cols].mean(axis=1, skipna=True)

_fd, _tmp = tempfile.mkstemp(suffix='.parquet')
os.close(_fd)
target_series.to_frame(name='TARGET_4_10').to_parquet(_tmp, index=False)
data = data.cbind(h2o.import_file(_tmp))
os.unlink(_tmp)

print(f'Training samples with valid target: {target_series.notna().sum()}')

---
## AutoML Training

In [ ]:
# D28 features allowed for durability — exclude only D365 columns
y = 'TARGET_4_10'
x = [c for c in data.columns
     if not c.endswith('_d365')
     and c != 'participant_id' and c != y]

train = data[data[y].isna() == 0]
print(f'Training samples: {train.nrows}  |  Features: {len(x)}')

aml = H2OAutoML(max_models=10, seed=1, nfolds=5,
                keep_cross_validation_predictions=True,
                max_runtime_secs=AUTO_ML_MAX_RUNTIME_SECONDS)

_buf = io.StringIO()
with redirect_stdout(_buf), redirect_stderr(_buf):
    aml.train(x=x, y=y, training_frame=train)
print('Training complete.')

In [ ]:
lb = aml.leaderboard
print(lb.head(rows=lb.nrows))

In [ ]:
cv_preds = aml.leader.cross_validation_holdout_predictions().as_data_frame()['predict']
actuals = train[y].as_data_frame()[y]
rho, pval = spearmanr(actuals, cv_preds)
print(f'Task 4.10 — Spearman (5-fold CV): {rho:.3f}  (p={pval:.4f})')

In [ ]:
print(f'Leader model: {aml.leader.model_id}')
varimp = aml.leader.varimp(use_pandas=True)
display(varimp.head(20))
aml.leader.varimp_plot(num_of_features=20)

In [ ]:
challenge_hf = h2o.H2OFrame(challenge_data)
y_pred = aml.leader.predict(challenge_hf).as_data_frame()['predict']

results = pd.DataFrame({
    'Participant_ID': challenge_data['participant_id'].values,
    'Task_4.10': np.exp2(y_pred),
})
results.to_csv(f'{SUBMISSION_PATH}/task_4_10.csv', index=False)
results

In [ ]:
h2o.cluster().shutdown()

---
## Conclusion

- **Leader model:** (fill after run)
- **CV Spearman:** (fill after run)

**Target:** log2(geomean) of vaccine strain HAI titers at D365. In practice collapses to Vic B/Austria/1359417/2021 D365 only, since the other vaccine strains have no D365 measurements in training data — making this effectively identical to Task 4.9.

D28 features are included as predictors (durability task allowance), which gives this model more signal than the D28 tasks.

Submission saved to `automl_submission/task_4_10.csv` (raw titer scale via `np.exp2`).